In [129]:
from multiprocessing.connection import answer_challenge

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
# from Langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain. tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage
import base64
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

True

In [130]:
from langchain.tools import tool

@tool
def diet_recommendation(parameter: str, status: str) -> str:
    """Gives diet advice based on a lab parameter and whether it's high or low.

    Example: parameter="cholesterol", status="high"
    """

    advice = {
        ("cholesterol", "high"): "Avoid fried food. Eat more fiber, nuts, and fish.",
        ("hemoglobin", "low"): "Eat spinach, eggs, and red meat. Pair with citrus fruits.",
        ("sugar", "high"): "Cut down on sweets and rice. Eat more vegetables.",
        ("vitamin d", "low"): "Get more sunlight. Eat egg yolk and fatty fish.",
    }

    key = (parameter.lower(), status.lower())
    return advice.get(key, "No specific advice available. Please consult a doctor.")

In [131]:
with open("D:/langchain/7_multi_model/blood_work.png", "rb") as file:
    image_base64 = base64.b64encode(file.read()).decode()
image_base64 [:]

'...'

In [132]:
@tool
def get_blood_report() -> AIMessage:
    """Returns the extracted contents of the user's uploaded blood report.
    Call this first whenever you need to know the report's test results
    before giving diet, exercise, or urgent-care advice.
    """
    return response   # the text we extracted earlier using vision

In [133]:
llm = init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct")
message = HumanMessage(content=[
          {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
          {"type": "text",    "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])


response = llm.invoke([message])

print(response.content)



**Extracted Test Results and Abnormal Values**

The following are the extracted test results with abnormal values flagged:

### Complete Blood Count (CBC)

* Hemoglobin: **15.1 g/dL** (Normal: 13.5-17.5) - **Within Normal Range**
* Hematocrit: **44%** (Normal: 41-53) - **Within Normal Range**
* WBC: **6.8 x10^3/uL** (Normal: 4.5-11.0) - **Within Normal Range**
* Platelets: **220 x10^3/uL** (Normal: 150-400) - **Within Normal Range**

### Lipid Panel

* Total Cholesterol: **238 mg/dL** (Normal: <200) - **ABNORMAL (HIGH)**
* LDL Cholesterol: **162 mg/dL** (Normal: <100) - **ABNORMAL (HIGH)**
* HDL Cholesterol: **36 mg/dL** (Normal: >40) - **ABNORMAL (LOW)**
* Triglycerides: **188 mg/dL** (Normal: <150) - **ABNORMAL (HIGH)**

### Metabolic Panel

* Glucose (Fasting): **92 mg/dL** (Normal: 70-99) - **Within Normal Range**
* HbA1c: **5.3%** (Normal: <5.7) - **Within Normal Range**
* Creatinine: **1.0 mg/dL** (Normal: 0.7-1.3) - **Within Normal Range**
* eGFR: **82 mL/min** (Normal: >60) - *

In [134]:
agent = create_agent(
    llm,
    tools=[diet_recommendation,get_blood_report],
    system_prompt="You are a helpful lab assistant for an health report 5.",
    checkpointer= InMemorySaver()
)

In [135]:
def ask2(question: str) -> str:
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent.invoke({"messages":[{"role":"user","content":question}]},config)
    print(result["messages"][-1].content)

In [136]:
ask2("can you see my reports")

Your blood report shows the following abnormal values:

* Total Cholesterol: 238 mg/dL (high)
* LDL Cholesterol: 162 mg/dL (high)
* HDL Cholesterol: 36 mg/dL (low)
* Triglycerides: 188 mg/dL (high)

Based on these results, I can provide you with diet recommendations. Would you like me to suggest a diet plan?


In [137]:
ask2("what is your previous answer")

Your blood report shows the following abnormal values:

* Total Cholesterol: 238 mg/dL (high)
* LDL Cholesterol: 162 mg/dL (high)
* HDL Cholesterol: 36 mg/dL (low)
* Triglycerides: 188 mg/dL (high)

Based on these results, I can provide you with diet recommendations. Would you like me to suggest a diet plan?


In [138]:
ask2("Which of my results are outside the normal range?")


Based on your blood report, the following results are outside the normal range:

* Total Cholesterol: 238 mg/dL (high) 
* LDL Cholesterol: 162 mg/dL (high) 
* HDL Cholesterol: 36 mg/dL (low) 
* Triglycerides: 188 mg/dL (high) 

These values are abnormal and may require attention. I can provide you with diet recommendations to help manage these values. Would you like me to suggest a diet plan?


In [139]:
ask2("What should I worry about in my report, and what diet changes should I make?")


To manage your abnormal blood report results, consider the following diet changes:

* For high cholesterol:
	+ Avoid fried foods
	+ Eat more fiber-rich foods like fruits, vegetables, and whole grains
	+ Include healthy fats like nuts and seeds in your diet
* For high LDL Cholesterol:
	+ Consult a doctor for personalized advice
* For low HDL Cholesterol:
	+ Consult a doctor for personalized advice
* For high Triglycerides:
	+ Consult a doctor for personalized advice

Additionally, consider incorporating foods that are rich in omega-3 fatty acids, such as fatty fish, into your diet. It's also essential to maintain a balanced diet and a healthy lifestyle to manage your blood report results. Consult a doctor or a registered dietitian for personalized advice.
